# DIPLOMADO DE VISIÓN ARTIFICIAL — Dive In Learning
## MÓDULO 3: Transferencia de Aprendizaje (Transfer Learning)
---

**Autor del material pedagógico:** Dive In Learning  
**Librerías principales:** TensorFlow / Keras, OpenCV, NumPy, Matplotlib

### ¿QUÉ APRENDERÁS EN ESTE MÓDULO?
1. **Modelos CNN preentrenados**
   - ResNet
   - VGG16 y VGG19
   - YOLO
   - Fine Tuning (ajuste fino)
2. **Clasificadores en Cascada**
   - Creación de datasets de entrenamiento
   - Personalización de los modelos

### ¿CÓMO USAR ESTE NOTEBOOK?
Lee cada sección en orden. Cada bloque de código incluye:
- Una explicación del **CONCEPTO** teórico (en las celdas de texto).
- Comentarios línea por línea explicando la **SINTAXIS**.
- Notas de *"¿Por qué hacemos esto?"* para que entiendas el razonamiento.

### REQUISITOS PREVIOS:
Asegúrate de instalar las siguientes librerías antes de comenzar:
```bash
pip install tensorflow opencv-python matplotlib numpy pillow
```

## SECCIÓN 0 — IMPORTACIONES GENERALES

Antes de programar, necesitamos "traer" las herramientas que vamos a usar. En Python, esto se hace con la palabra clave `import`.

Piensa en las librerías como **"cajas de herramientas especializadas"**:
- **TensorFlow/Keras** → caja de herramientas para redes neuronales.
- **OpenCV (cv2)** → caja de herramientas para procesamiento de imágenes.
- **NumPy** → caja de herramientas para cálculos numéricos.
- **Matplotlib** → caja de herramientas para graficar y visualizar.

In [ ]:
import numpy as np                          # Manejo eficiente de arreglos numéricos (matrices, vectores)
import matplotlib.pyplot as plt             # Para graficar resultados y visualizar imágenes
import cv2                                  # OpenCV: procesamiento de imágenes y video
import os                                   # Para trabajar con rutas de archivos y carpetas
import warnings                             # Para suprimir advertencias no críticas
warnings.filterwarnings('ignore')           # Ocultamos advertencias para que la salida sea más limpia

# --- Importaciones de TensorFlow y Keras ---
# TensorFlow es el framework de Deep Learning que usaremos.
# Keras es la API de alto nivel que viene integrada en TensorFlow,
# y nos permite construir redes neuronales con muy pocas líneas de código.

import tensorflow as tf                     # El framework principal de Deep Learning

# Importamos modelos preentrenados famosos directamente desde Keras
from tensorflow.keras.applications import (
    ResNet50,                               # Red profunda con conexiones residuales (Microsoft, 2015)
    VGG16,                                  # Red clásica de Oxford (2014), 16 capas
    VGG19                                   # Versión más profunda de VGG, 19 capas
)

# Importamos capas para construir o modificar redes neuronales
from tensorflow.keras.layers import (
    Dense,                                  # Capa "densa" o completamente conectada (fully connected)
    GlobalAveragePooling2D,                 # Reduce el mapa de características a un solo vector promedio
    Flatten,                                # Aplana (convierte) una matriz 3D en un vector 1D
    Dropout                                 # Apaga neuronas aleatoriamente para evitar overfitting
)

from tensorflow.keras.models import Model  # Clase base para construir modelos personalizados
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator,                     # Genera y aumenta imágenes para entrenamiento
    load_img,                               # Carga una imagen desde disco
    img_to_array                            # Convierte una imagen PIL a un arreglo NumPy
)

# Preprocesamiento específico de cada arquitectura
# Cada red fue entrenada con imágenes normalizadas de forma diferente,
# por eso cada una tiene su propia función de preprocesamiento.
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.vgg16    import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.resnet50 import decode_predictions  # Convierte predicciones a etiquetas legibles

print("✅ Todas las librerías se importaron correctamente.")
print(f"   TensorFlow versión: {tf.__version__}")

## SECCIÓN 1 — ¿QUÉ ES LA TRANSFERENCIA DE APRENDIZAJE?

### CONCEPTO CLAVE:
Imagina que quieres aprender a jugar ajedrez. Si ya sabes jugar damas, no empiezas desde cero — reutilizas tu conocimiento sobre estrategias, movimientos, anticipar al oponente, etc.

La **Transferencia de Aprendizaje (Transfer Learning)** es exactamente eso: tomar una red neuronal que YA fue entrenada con millones de imágenes (como ImageNet, con 1,000 categorías) y REUTILIZAR ese conocimiento para resolver un problema nuevo con MUCHO menos datos y tiempo.

### ¿POR QUÉ ES TAN PODEROSO?
Entrenar una red como ResNet desde cero requiere:
- Millones de imágenes etiquetadas
- Semanas de entrenamiento en GPUs potentes
- Mucho dinero en cómputo

Con Transfer Learning podemos:
- Usar CIENTOS de imágenes propias
- Entrenar en MINUTOS (incluso en CPU)
- Obtener resultados muy precisos

### FLUJO GENERAL:
1. Tomas el **Modelo preentrenado**.
2. **Congelas sus pesos** (para que retenga lo que ya sabe: detectar bordes, texturas, formas).
3. **Agregas tus capas** (para que aprenda TU clasificación específica, ej: gatos vs perros).

## SECCIÓN 12 — MODELOS CNN PREENTRENADOS

### 12.1 — ResNet50

**CONCEPTO:**
ResNet (Residual Network) fue creada por Microsoft en 2015 y ganó el concurso ImageNet con un error menor al 3.6% (¡mejor que humanos!).

Su innovación clave son las **"conexiones residuales"** o **"skip connections"**. En lugar de que cada capa solo procese la salida de la capa anterior, ResNet suma también la entrada original. Esto permite entrenar redes MUY profundas (50, 101, 152 capas) sin que el gradiente "desaparezca" durante el entrenamiento.

**Arquitectura simplificada:**
```text
┌─────────────────────────────────────────────────────────────────┐
│  Entrada → Conv → BatchNorm → ReLU → Conv → BatchNorm           │
│                                                    +  ←(skip)   │
│                                               → ReLU → Salida   │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
print("\n--- 12.1 ResNet50 ---")

# ── Cargando ResNet50 preentrenada ──────────────────────────────────────────
# weights='imagenet' → Usamos los pesos entrenados con el dataset ImageNet
#                       (1.2 millones de imágenes, 1000 categorías)
# include_top=True  → Incluimos la cabeza clasificadora original (1000 clases)
# La primera vez que ejecutes esto, TensorFlow descargará los pesos (~100 MB)

modelo_resnet = ResNet50(
    weights='imagenet',     # Pesos preentrenados en ImageNet
    include_top=True        # Incluye las capas finales de clasificación
)

# Mostramos un resumen de la arquitectura
print("\n📋 Arquitectura ResNet50 (resumen compacto):")
total_params = modelo_resnet.count_params()
print(f"   Número total de parámetros: {total_params:,}")


# ── Usando ResNet50 para hacer predicciones (Inferencia) ───────────────────
print("\n🔍 Ejemplo de predicción con ResNet50:")

# Creamos una imagen de ejemplo con valores aleatorios (solo para demostración)
imagen_ejemplo = np.random.randint(
    low=0,          # Valor mínimo de píxel
    high=256,       # Valor máximo de píxel (256 es exclusivo → máximo 255)
    size=(224, 224, 3),  # Forma: (alto, ancho, canales)
    dtype=np.uint8  # Tipo entero sin signo de 8 bits (0 a 255)
)

# Las redes neuronales esperan un "batch" (lote) de imágenes, no una sola.
imagen_batch = np.expand_dims(imagen_ejemplo, axis=0)

# Preprocesamos la imagen con la función específica de ResNet.
imagen_preprocesada = resnet_preprocess(imagen_batch.astype('float32'))

# model.predict() ejecuta la red neuronal con nuestra imagen
predicciones = modelo_resnet.predict(imagen_preprocesada, verbose=0)

# decode_predictions convierte los índices numéricos a nombres legibles
resultado = decode_predictions(predicciones, top=3)[0]

print("   Top 3 predicciones (imagen aleatoria, resultados no tienen sentido real):")
for i, (codigo_clase, nombre_clase, probabilidad) in enumerate(resultado):
    print(f"   {i+1}. {nombre_clase:<20} Probabilidad: {probabilidad:.4f} ({probabilidad*100:.2f}%)")


# ── Función reutilizable para predecir con ResNet ──────────────────────────
def predecir_con_resnet(ruta_imagen):
    """
    Carga una imagen desde disco, la preprocesa y obtiene predicciones
    usando ResNet50 preentrenada en ImageNet.
    """
    img = load_img(ruta_imagen, target_size=(224, 224))
    img_array = img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_preprocesada = resnet_preprocess(img_array)
    preds = modelo_resnet.predict(img_preprocesada, verbose=0)
    return decode_predictions(preds, top=5)[0]

# Ejemplo de uso (descomenta si tienes una imagen):
# resultados = predecir_con_resnet("perro.jpg")
# for codigo, nombre, prob in resultados:
#     print(f"{nombre}: {prob*100:.1f}%")

### 12.2 — VGG16 y VGG19

**CONCEPTO:**
VGG fue desarrollada por el Visual Geometry Group de Oxford en 2014. Su filosofía es la simplicidad: usa SOLO convoluciones pequeñas (3×3) apiladas en profundidad. Esto la hace muy simple de entender y adaptar.

- **VGG16** → 16 capas con pesos entrenables (13 conv + 3 densas)
- **VGG19** → 19 capas con pesos entrenables (16 conv + 3 densas)

**Arquitectura de VGG16 (simplificada):**
```text
┌──────────────────────────────────────────────────────────────┐
│  Entrada(224×224×3)                                          │
│  → Bloque1: Conv3×3 × 2  → MaxPool                           │
│  → Bloque2: Conv3×3 × 2  → MaxPool                           │
│  → Bloque3: Conv3×3 × 3  → MaxPool                           │
│  → Bloque4: Conv3×3 × 3  → MaxPool                           │
│  → Bloque5: Conv3×3 × 3  → MaxPool                           │
│  → Flatten → Dense(4096) → Dense(4096) → Dense(1000)         │
└──────────────────────────────────────────────────────────────┘
```

**DIFERENCIA CLAVE entre VGG y ResNet:**
- **VGG:** arquitectura lineal, sencilla, más fácil de entender.
- **ResNet:** conexiones residuales, más profunda, más eficiente.

In [ ]:
print("\n--- 12.2 VGG16 y VGG19 ---")

# ── Cargando VGG16 SIN la cabeza clasificadora ────────────────────────────
# include_top=False → Excluimos las capas Dense finales
#                     Esto es lo que queremos para Transfer Learning:
#                     usamos VGG como "extractor de características"
#                     y luego agregamos NUESTRAS propias capas de clasificación.

modelo_vgg16_base = VGG16(
    weights='imagenet',         # Pesos preentrenados de ImageNet
    include_top=False,          # Sin las capas densas finales (sin "cabeza")
    input_shape=(224, 224, 3)   # Tamaño de imagen de entrada
)

print(f"\n📋 VGG16 (extractor de características, sin cabeza clasificadora):")
print(f"   Parámetros totales: {modelo_vgg16_base.count_params():,}")

# ── Cargando VGG19 ────────────────────────────────────────────────────────
modelo_vgg19_base = VGG19(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
print(f"\n📋 VGG19 (extractor de características, sin cabeza clasificadora):")
print(f"   Parámetros totales: {modelo_vgg19_base.count_params():,}")

# ── Comparativa visual ────────────────────────────────────────────────────
print("\n📊 Comparativa de modelos preentrenados:")
print(f"   {'Modelo':<12} {'Parámetros':<20} {'Profundidad':<15} {'Precisión en ImageNet'}")
print(f"   {'-'*65}")
print(f"   {'VGG16':<12} {'138,357,544':<20} {'16 capas':<15} {'~92.7% (top-5)'}")
print(f"   {'VGG19':<12} {'143,667,240':<20} {'19 capas':<15} {'~93.1% (top-5)'}")
print(f"   {'ResNet50':<12} {'25,636,712':<20} {'50 capas':<15} {'~93.3% (top-5)'}")

### 12.3 — YOLO (You Only Look Once)

**CONCEPTO:**
YOLO es un algoritmo de **DETECCIÓN DE OBJETOS**, no solo clasificación. La diferencia es crucial:

- **Clasificación:** "Esta imagen contiene un GATO"
- **Detección:** "Hay un GATO en la posición (x=120, y=45, ancho=200, alto=180)"

**¿Por qué YOLO es especial?**
Antes de YOLO (2016), la detección requería dos pasos separados (muy lentos). YOLO lo hace en **UN SOLO PASO** (de ahí "You Only Look Once"):
- Divide la imagen en una cuadrícula (ej: 13×13)
- Para cada celda predice: ¿hay objeto? + qué es + dónde está exactamente
- Resultado: muy rápido (puede procesar video en tiempo real)

**FUNCIONAMIENTO INTERNO:**
```text
┌────────────────────────────────────────────────────────────────┐
│  Imagen → Red CNN → Grid de predicciones                       │
│                                                                │
│  Para cada celda del grid:                                     │
│    - Bounding boxes (x, y, ancho, alto)                        │
│    - Objectness score (¿hay objeto aquí?)                      │
│    - Class probabilities (¿qué objeto es?)                     │
└────────────────────────────────────────────────────────────────┘
```

*Nota: A continuación simulamos su comportamiento y mostramos cómo usar la librería oficial `ultralytics`.*

In [ ]:
print("\n--- 12.3 YOLO (Detección de Objetos) ---")

def demostrar_yolo_con_opencv(imagen_np):
    """
    Demuestra los conceptos de YOLO usando OpenCV para simular
    el proceso de detección cuando ultralytics no está disponible.
    """
    print("\n   💡 Concepto YOLO: División en cuadrícula (Grid)")
    
    alto, ancho = imagen_np.shape[:2]
    
    # YOLO divide la imagen en celdas de cuadrícula
    tamaño_grid = 3                     # Dividiremos la imagen en 3×3 = 9 celdas
    alto_celda = alto // tamaño_grid    # // es división entera
    ancho_celda = ancho // tamaño_grid
    
    print(f"   Imagen de {ancho}×{alto} px dividida en grid {tamaño_grid}×{tamaño_grid}")
    print(f"   Tamaño de cada celda: {ancho_celda}×{alto_celda} px")
    
    print("\n   Simulando predicciones YOLO (valores aleatorios):")
    np.random.seed(42)
    
    for fila in range(tamaño_grid):
        for col in range(tamaño_grid):
            confianza = np.random.random()
            if confianza > 0.7:
                cx = np.random.random() 
                cy = np.random.random() 
                bw = np.random.random() * 0.8 + 0.2
                bh = np.random.random() * 0.8 + 0.2
                
                clases = ["persona", "auto", "perro", "gato", "bicicleta"]
                clase_detectada = np.random.choice(clases)
                
                print(f"   ✓ Celda ({fila},{col}): '{clase_detectada}' "
                      f"| Confianza: {confianza:.2f} "
                      f"| Centro: ({cx:.2f}, {cy:.2f}) "
                      f"| Tamaño: {bw:.2f}×{bh:.2f}")

# Creamos una imagen de prueba
imagen_demo = np.random.randint(0, 256, (416, 416, 3), dtype=np.uint8)
demostrar_yolo_con_opencv(imagen_demo)

# ── Código para usar YOLO real (si tienes ultralytics instalado) ─────────
print("\n   📝 Código para YOLO real (ejecutar si tienes: pip install ultralytics):")
codigo_yolo_real = """
    from ultralytics import YOLO
    import cv2
    
    # Cargar modelo YOLOv8 nano
    modelo_yolo = YOLO('yolov8n.pt')
    
    # Detectar objetos en una imagen
    resultados = modelo_yolo('mi_imagen.jpg')
    
    # Mostrar imagen con detecciones dibujadas
    resultado_imagen = resultados[0].plot()
    cv2.imshow('Detecciones YOLO', resultado_imagen)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
"""
print(codigo_yolo_real)

### 12.4 — FINE TUNING (Ajuste Fino)

**CONCEPTO:**
El Fine Tuning es la técnica más poderosa de Transfer Learning. Va más allá de simplemente agregar capas nuevas al final: también "descongela" algunas capas del modelo preentrenado y las re-entrena con nuestra data específica.

**Tres estrategias principales:**

**ESTRATEGIA 1: Feature Extraction (Extracción de características)**
→ Congelas TODAS las capas preentrenadas.
→ Solo entrenas las capas nuevas que agregas.
→ *Ideal para: datasets pequeños o muy similares a ImageNet.*

**ESTRATEGIA 2: Fine Tuning Parcial**
→ Congelas las capas iniciales (detectan bordes, texturas).
→ Descongelas las capas finales del modelo base (patrones complejos).
→ Entrenas capas descongeladas + capas nuevas.
→ *Recomendado para la mayoría de casos.*

**ESTRATEGIA 3: Fine Tuning Total**
→ Descongelas TODAS las capas.
→ Entrenas todo el modelo con TU data.
→ *Requiere dataset gigante y mucho tiempo.*

**ANALOGÍA:**
Un chef experto (modelo preentrenado) sabe técnicas básicas de cocina. Si quieres que aprenda cocina mexicana:
- *Feature Extraction*: Le enseñas solo recetas mexicanas.
- *Fine Tuning Parcial*: Le enseñas recetas + ajustas sus técnicas de sazón.
- *Fine Tuning Total*: Le enseñas TODO desde cero pero enfocado a México.

In [ ]:
print("\n--- 12.4 Fine Tuning ---")

def construir_modelo_con_transfer_learning(
        num_clases,
        arquitectura='vgg16',
        estrategia='feature_extraction',
        capas_a_descongelar=5
):
    print(f"\n   🔨 Construyendo modelo con Transfer Learning...")
    print(f"      Arquitectura base: {arquitectura.upper()}")
    print(f"      Estrategia: {estrategia}")
    
    # ── PASO 1: Seleccionar y cargar el modelo base ──────────────────────
    modelos_disponibles = {
        'vgg16':   VGG16,
        'vgg19':   VGG19,
        'resnet50': ResNet50
    }
    
    if arquitectura not in modelos_disponibles:
        raise ValueError(f"Arquitectura no disponible.")
    
    ClaseModelo = modelos_disponibles[arquitectura]
    modelo_base = ClaseModelo(
        weights='imagenet',
        include_top=False,      # Sin cabeza: usaremos la nuestra
        input_shape=(224, 224, 3)
    )
    
    # ── PASO 2: Congelar el modelo base (siempre al inicio) ──────────────
    modelo_base.trainable = False
    
    # ── PASO 3: Aplicar estrategia de Fine Tuning (si se solicitó) ───────
    if estrategia == 'fine_tuning':
        total_capas = len(modelo_base.layers)
        capa_inicio_descongelado = total_capas - capas_a_descongelar
        
        for capa in modelo_base.layers[capa_inicio_descongelado:]:
            if not isinstance(capa, tf.keras.layers.BatchNormalization):
                capa.trainable = True   # Descongela esta capa
    
    # ── PASO 4: Construir NUESTRA cabeza clasificadora ────────────────────
    salida_base = modelo_base.output
    
    # GlobalAveragePooling2D: Promedia los mapas de características para aplanarlos
    x = GlobalAveragePooling2D()(salida_base)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.5)(x) # Previene el overfitting
    
    if num_clases == 2:
        salida = Dense(1, activation='sigmoid')(x)
    else:
        salida = Dense(num_clases, activation='softmax')(x)
    
    # ── PASO 5: Crear el modelo final ─────────────────────────────────────
    modelo_final = Model(inputs=modelo_base.input, outputs=salida)
    
    # ── PASO 6: Compilar el modelo ────────────────────────────────────────
    modelo_final.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=1e-4 if estrategia == 'feature_extraction' else 1e-5
        ),
        loss='binary_crossentropy' if num_clases == 2 else 'sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    total = modelo_final.count_params()
    entrenables = sum(tf.size(w).numpy() for w in modelo_final.trainable_weights)
    congelados = total - entrenables
    
    print(f"\n   📊 Estadísticas del modelo construido:")
    print(f"      Parámetros entrenables:    {entrenables:>12,}  ← estos aprenderán")
    print(f"      Parámetros congelados:     {congelados:>12,}  ← estos NO cambian")
    
    return modelo_final

# ── Construyendo modelos con diferentes estrategias ────────────────────────
print("\n🎯 ESTRATEGIA 1: Feature Extraction con VGG16 (2 clases: gato vs perro)")
modelo_extraccion = construir_modelo_con_transfer_learning(
    num_clases=2, arquitectura='vgg16', estrategia='feature_extraction'
)

print("\n🎯 ESTRATEGIA 2: Fine Tuning con ResNet50 (5 clases: flores)")
modelo_fine_tuning = construir_modelo_con_transfer_learning(
    num_clases=5, arquitectura='resnet50', estrategia='fine_tuning', capas_a_descongelar=10
)

## SECCIÓN 13 — CLASIFICADORES EN CASCADA

### CONCEPTO:
Los clasificadores en cascada son un método CLÁSICO (antes de las CNN modernas) para detección de objetos. Son extremadamente rápidos y funcionan en tiempo real incluso en hardware limitado (Raspberry Pi, microcontroladores).

**¿CÓMO FUNCIONAN?**
Imagina una serie de "filtros" (etapas) cada vez más estrictos:
```text
Región candidata → [Filtro 1: ¿hay bordes?]
                         ↓ SI pasa
                   [Filtro 2: ¿hay ojos?]
                         ↓ SI pasa
                   [Filtro 3: ¿hay nariz y boca?]
                         ↓ SI pasa
                   "¡ES UNA CARA!"
```
Si una región NO pasa cualquier filtro → se descarta inmediatamente (CASCADA).

**HAAR CASCADE vs. CNN:**
- **Haar Cascade**: ✅ Extremadamente rápido ✅ No requiere GPU ❌ Menos preciso.
- **CNN (YOLO, etc)**: ✅ Muy preciso ✅ Maneja variaciones (ángulos, luz) ❌ Más lento.

### 13.1 — Creación de Datasets de Entrenamiento

Para entrenar un modelo personalizado necesitamos datos. La cantidad y calidad de tus datos determina qué tan bien funciona tu modelo.

**REGLAS DE ORO:**
1. **BALANCE:** Misma cantidad de imágenes por clase.
2. **VARIEDAD:** Incluye diferentes condiciones (ángulos, luces, fondos).
3. **CALIDAD:** Imágenes nítidas y bien etiquetadas.

**ESTRUCTURA RECOMENDADA:**
```text
dataset/
├── train/           ← 70-80% de los datos (para entrenar)
│   ├── clase_A/
│   └── clase_B/
├── validation/      ← 10-15% de los datos (evaluación DURANTE entrenamiento)
└── test/            ← 10-15% de los datos (evaluación AL FINAL)
```

In [ ]:
print("\n--- 13.1 Creación de Datasets de Entrenamiento ---")

def crear_estructura_dataset(ruta_base, clases, splits=None):
    """Crea la estructura de carpetas para un dataset de entrenamiento."""
    if splits is None:
        splits = {'train': 0.70, 'validation': 0.15, 'test': 0.15}
    
    rutas_creadas = {}
    print(f"\n   📁 Creando estructura de dataset en: '{ruta_base}'")
    
    for split_nombre in splits.keys():
        for clase in clases:
            ruta_completa = os.path.join(ruta_base, split_nombre, clase)
            os.makedirs(ruta_completa, exist_ok=True)
            clave = f"{split_nombre}/{clase}"
            rutas_creadas[clave] = ruta_completa
            print(f"   ✓ Creada: {ruta_completa}")
    
    return rutas_creadas

rutas = crear_estructura_dataset(
    ruta_base="./dataset_ejemplo",
    clases=["gato", "perro", "pajaro"]
)

# ── DATA AUGMENTATION: Multiplicar datos existentes ───────────────────────
# Genera variaciones artificiales (voltear, rotar, zoom) para evitar overfitting.

print("\n   🔄 Configurando Data Augmentation:")
generador_entrenamiento = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# El generador de validación SOLO normaliza, SIN transformaciones
generador_validacion = ImageDataGenerator(rescale=1.0/255.0)

def cargar_dataset_desde_carpetas(ruta_train, ruta_val, tamaño_imagen=224, batch_size=32):
    """Carga imágenes desde carpetas y devuelve generadores."""
    train_gen = generador_entrenamiento.flow_from_directory(
        ruta_train, target_size=(tamaño_imagen, tamaño_imagen),
        batch_size=batch_size, class_mode='categorical', shuffle=True
    )
    val_gen = generador_validacion.flow_from_directory(
        ruta_val, target_size=(tamaño_imagen, tamaño_imagen),
        batch_size=batch_size, class_mode='categorical', shuffle=False
    )
    nombres_clases = {v: k for k, v in train_gen.class_indices.items()}
    return train_gen, val_gen, nombres_clases

print("   ✅ Generadores de data listos para usarse.")

### 13.2 — Personalización de los Modelos (Entrenamiento)

**PIPELINE COMPLETO DE TRANSFER LEARNING:**
1. Definir el problema (clases).
2. Recolectar/organizar datos.
3. Elegir arquitectura base (VGG, ResNet).
4. Construir la cabeza clasificadora.
5. Entrenar fase 1: Feature Extraction.
6. Entrenar fase 2: Fine Tuning (opcional).
7. Evaluar y Guardar.

In [ ]:
print("\n--- 13.2 Entrenamiento del Modelo ---")

def entrenar_modelo(modelo, train_generator, val_generator, epochs=10, callbacks_adicionales=None):
    """Entrena el modelo usando los generadores de datos."""
    print(f"\n   🚀 Iniciando entrenamiento por {epochs} epochs...")
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
        tf.keras.callbacks.ModelCheckpoint(filepath='./mejor_modelo.keras', monitor='val_accuracy', save_best_only=True)
    ]
    
    if callbacks_adicionales: callbacks.extend(callbacks_adicionales)
    
    historial = modelo.fit(
        train_generator, epochs=epochs,
        validation_data=val_generator, callbacks=callbacks, verbose=1
    )
    return historial

def visualizar_historial_entrenamiento(historial, titulo="Entrenamiento"):
    """Grafica las curvas de pérdida y precisión."""
    metricas_disponibles = list(historial.history.keys())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(titulo, fontsize=16, fontweight='bold')
    
    epochs_rango = range(1, len(historial.history['loss']) + 1)
    
    # Pérdida (Loss)
    ax1.plot(epochs_rango, historial.history['loss'], 'b-o', label='Entrenamiento')
    if 'val_loss' in metricas_disponibles:
        ax1.plot(epochs_rango, historial.history['val_loss'], 'r--s', label='Validación')
    ax1.set_title('Curva de Pérdida')
    ax1.legend(); ax1.grid(True, alpha=0.3)
    
    # Precisión (Accuracy)
    ax2.plot(epochs_rango, historial.history['accuracy'], 'b-o', label='Entrenamiento')
    if 'val_accuracy' in metricas_disponibles:
        ax2.plot(epochs_rango, historial.history['val_accuracy'], 'r--s', label='Validación')
    ax2.set_title('Curva de Precisión')
    ax2.legend(); ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ── Simulación de historial para demostración pedagógica ──────────────────
class HistorialSimulado:
    def __init__(self):
        epochs = 15
        np.random.seed(123)
        self.history = {
            'loss': [max(0.05, x + np.random.normal(0, 0.03)) for x in np.linspace(1.5, 0.15, epochs)],
            'val_loss': [max(0.08, x + np.random.normal(0, 0.05)) for x in np.linspace(1.6, 0.25, epochs)],
            'accuracy': [min(0.99, x + np.random.normal(0, 0.02)) for x in np.linspace(0.40, 0.95, epochs)],
            'val_accuracy': [min(0.97, x + np.random.normal(0, 0.03)) for x in np.linspace(0.35, 0.88, epochs)]
        }

visualizar_historial_entrenamiento(HistorialSimulado(), titulo="Entrenamiento Exitoso (Simulación)")

## PIPELINE COMPLETO: PREDICCIÓN CON MODELO ENTRENADO

Una vez que tienes el modelo entrenado (y guardado), puedes usarlo para hacer predicciones en nuevas imágenes. A esto se le llama **Inferencia**.

**Flujo:** `Imagen en disco → Cargar → Redimensionar → Normalizar → modelo.predict()`

In [ ]:
def predecir_imagen_personalizada(modelo, ruta_imagen, nombres_clases, tamaño=224):
    img = load_img(ruta_imagen, target_size=(tamaño, tamaño))
    img_array = img_to_array(img) / 255.0  # ¡Misma normalización que el entrenamiento!
    img_batch = np.expand_dims(img_array, axis=0)
    
    probabilidades = modelo.predict(img_batch, verbose=0)[0]
    indice_prediccion = np.argmax(probabilidades)
    clase_predicha = nombres_clases.get(indice_prediccion, f"clase_{indice_prediccion}")
    confianza = probabilidades[indice_prediccion]
    
    return clase_predicha, confianza, probabilidades

def guardar_modelo(modelo, ruta="./mi_modelo_final.keras"):
    modelo.save(ruta)
    print(f"   ✅ Modelo guardado en: {ruta}")

def cargar_modelo(ruta="./mi_modelo_final.keras"):
    modelo_cargado = tf.keras.models.load_model(ruta)
    print(f"   ✅ Modelo cargado desde: {ruta}")
    return modelo_cargado

print("\n💾 Guardando modelo de ejemplo:")
guardar_modelo(modelo_extraccion, "./modelo_vgg16_cats_dogs.keras")

## HAAR CASCADE — Clasificadores en Cascada con OpenCV

OpenCV viene con clasificadores preentrenados (XMLs) para detección de características comunes como Caras frontales, Ojos, Sonrisas y Cuerpo completo.

Utilizan una técnica de ventana deslizante y filtros rápidos para encontrar la característica.

In [ ]:
print("\n--- Haar Cascade: Detección en tiempo real con OpenCV ---")

def demostrar_haar_cascade():
    # Buscamos el archivo en la instalación de OpenCV
    ruta_cascada = cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
    
    if not os.path.exists(ruta_cascada):
        return print("   ⚠️  Archivo de cascada no encontrado.")
    
    detector_caras = cv2.CascadeClassifier(ruta_cascada)
    
    # Creamos una imagen sintética en escala de grises (requerido por Haar)
    imagen_prueba = np.random.randint(50, 200, (480, 640), dtype=np.uint8)
    
    # detectMultiScale: detecta objetos a múltiples escalas
    rostros = detector_caras.detectMultiScale(
        imagen_prueba,
        scaleFactor=1.1,     # Cuánto se reduce la imagen en cada escala
        minNeighbors=5,      # Cuántas detecciones confirman el rostro
        minSize=(30, 30)     # Tamaño mínimo
    )
    
    print(f"   ✅ Clasificador cargado.")
    print(f"   Rostros detectados en imagen aleatoria: {len(rostros)}")

demostrar_haar_cascade()

## RESUMEN FINAL Y GUÍA DE ESTUDIO
---
### 📌 12.1 ResNet50
- Arquitectura profunda con conexiones residuales (skip connections).
- Ideal para: imágenes con patrones complejos, clasificación general.

### 📌 12.2 VGG16/19
- Arquitectura simple y lineal, solo convoluciones 3×3.
- Ideal para: problemas donde la interpretabilidad importa.

### 📌 12.3 YOLO
- Detección de objetos en tiempo real (no solo clasificación).
- Un solo paso: localiza Y clasifica objetos simultáneamente.

### 📌 12.4 Fine Tuning
- **Feature Extraction:** congelas todo, solo entrenas capas nuevas.
- **Fine Tuning Parcial:** descongelas últimas capas del modelo base.

### 📌 13.1 Creación de Datasets
- Estructura: `dataset/train/` + `/validation/` + `/test/`
- División recomendada: 70% train, 15% val, 15% test.
- Mínimo recomendado: 200-500 imágenes por clase.

### 📌 13.2 Personalización
- Callbacks clave: `EarlyStopping`, `ReduceLROnPlateau`, `ModelCheckpoint`.
- Guardar: `modelo.save('mi_modelo.keras')`

---

### 🚀 PRÓXIMOS PASOS RECOMENDADOS
1. 📚 Estudia la arquitectura de ResNet dibujando su diagrama de bloques
2. 💻 Descarga el dataset *Cats vs Dogs* de Kaggle y aplica VGG16
3. 🔬 Experimenta con diferentes learning rates y observa el efecto
4. 📊 Practica leyendo e interpretando las curvas de entrenamiento

In [ ]:
print("✅ Módulo 3 completado. ¡Listo para practicar Transfer Learning!")

# Limpieza: eliminamos la carpeta de ejemplo
import shutil
if os.path.exists("./dataset_ejemplo"):
    shutil.rmtree("./dataset_ejemplo")